# Employee Burnout Prediction

Notebook ini dibuat untuk project Data Mining dengan framework **CRISP-DM**.

Versi ini dibuat lebih visual agar hasil analisis lebih mudah dipahami. Kodenya tetap dibuat sederhana dan beginner friendly, sehingga alurnya mudah diikuti.

Notebook ini tidak membuat folder baru dan tidak menyimpan file tambahan.

## Framework CRISP-DM

Tahapan yang digunakan:

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Data Clustering
5. Regression
6. Classification
7. Evaluation
8. Deployment sederhana

## 1. Business Understanding

Tujuan project ini adalah mengelompokkan karyawan berdasarkan risiko burnout.

Pada project ini:

- K-Means digunakan untuk membuat 2 cluster risiko burnout.
- Kolom `Cluster` hasil K-Means digunakan sebagai target untuk regression dan classification.
- Output akhir berupa predicted cluster, label risiko, dan rekomendasi sederhana.

## Import Library

Library yang digunakan adalah library dasar untuk data mining, machine learning, dan visualisasi.

In [ ]:
import os

# Mengurangi warning CPU pada beberapa environment.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("MPLCONFIGDIR", "assets/matplotlib_cache")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## Membuka Dataset

Di Google Colab, jalankan cell berikut lalu upload file dataset:

`employee_burnout_analysis.csv`

Jika dijalankan di laptop dari root project, notebook akan mencoba membaca file dari:

`data/raw/employee_burnout_analysis.csv`

In [ ]:
# Cara 1: Google Colab, upload file langsung.
# Cara 2: Jika tidak di Colab, gunakan path lokal default.

try:
    from google.colab import files
    uploaded = files.upload()
    dataset_path = list(uploaded.keys())[0]
except ModuleNotFoundError:
    dataset_path = "data/raw/employee_burnout_analysis.csv"

print("Dataset yang digunakan:", dataset_path)

## 2. Data Understanding

Pada tahap ini kita membaca dataset dan memahami isi datanya.

Yang ditampilkan:

- Nama variabel
- Jumlah variabel
- Jumlah record
- Tipe data
- Sifat data
- Missing values
- Grafik awal agar data lebih mudah dibaca

In [ ]:
# Dataset menggunakan separator titik koma (;) dan angka desimal menggunakan koma (,).
df = pd.read_csv(dataset_path, sep=";", decimal=",", encoding="utf-8-sig")

# Menampilkan 5 data pertama.
df.head()

In [ ]:
print("Nama variabel dataset:")
print(list(df.columns))

print()
print("Jumlah variabel:", df.shape[1])
print("Jumlah record:", df.shape[0])

In [ ]:
def cek_sifat_data(nama_kolom, data_kolom):
    if "date" in nama_kolom.lower():
        return "date"
    elif pd.api.types.is_numeric_dtype(data_kolom):
        return "numeric"
    elif data_kolom.nunique() > 30:
        return "string"
    else:
        return "categorical"

identifikasi_list = []

for kolom in df.columns:
    identifikasi_list.append({
        "Nama Variabel": kolom,
        "Tipe Data": str(df[kolom].dtype),
        "Sifat Data": cek_sifat_data(kolom, df[kolom]),
        "Missing Values": df[kolom].isna().sum(),
        "Jumlah Nilai Unik": df[kolom].nunique()
    })

identifikasi_data = pd.DataFrame(identifikasi_list)
identifikasi_data

### Grafik Missing Values

Grafik ini membantu melihat kolom mana yang memiliki data kosong.

In [ ]:
missing_data = df.isna().sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_data.index, y=missing_data.values)
plt.title("Jumlah Missing Values pada Setiap Kolom")
plt.xlabel("Nama Kolom")
plt.ylabel("Jumlah Missing Values")
plt.xticks(rotation=45, ha="right")
plt.show()

### Grafik Data Kategori

Grafik ini menunjukkan perbandingan jumlah data pada kolom kategori.

In [ ]:
kolom_kategori_awal = ["Gender", "Company Type", "WFH Setup Available"]

plt.figure(figsize=(15, 4))

for nomor, kolom in enumerate(kolom_kategori_awal):
    plt.subplot(1, 3, nomor + 1)
    sns.countplot(data=df, x=kolom)
    plt.title("Jumlah Data " + kolom)
    plt.xlabel(kolom)
    plt.ylabel("Jumlah")

plt.tight_layout()
plt.show()

## 3. Data Preparation

Pada tahap ini kita membersihkan data agar siap dipakai untuk machine learning.

Langkah yang dilakukan:

- Mengubah `Date of Joining` menjadi tipe tanggal
- Mengubah kolom angka menjadi numeric
- Membuat kolom baru `Years Working`
- Mengisi missing values
- Memilih 4 variabel numeric untuk modeling

In [ ]:
df_clean = df.copy()

# Mengubah Date of Joining menjadi tipe tanggal.
df_clean["Date of Joining"] = pd.to_datetime(
    df_clean["Date of Joining"],
    format="%d/%m/%y",
    errors="coerce"
)

# Mengubah kolom numerik menjadi numeric.
kolom_numerik = [
    "Designation",
    "Resource Allocation",
    "Mental Fatigue Score",
    "Burn Rate"
]

for kolom in kolom_numerik:
    df_clean[kolom] = pd.to_numeric(df_clean[kolom], errors="coerce")

# Membuat kolom Years Working.
tahun_terbaru = df_clean["Date of Joining"].dt.year.max()
df_clean["Years Working"] = tahun_terbaru - df_clean["Date of Joining"].dt.year

df_clean.head()

In [ ]:
# Melihat missing values sebelum diisi.
df_clean.isna().sum()

In [ ]:
# Memilih minimal 4 variabel numeric untuk clustering, regression, dan classification.
fitur = [
    "Designation",
    "Resource Allocation",
    "Mental Fatigue Score",
    "Years Working"
]

print("Fitur yang digunakan:")
print(fitur)

In [ ]:
# Mengisi missing values pada kolom numeric dengan median.
for kolom in fitur + ["Burn Rate"]:
    df_clean[kolom] = df_clean[kolom].fillna(df_clean[kolom].median())

# Mengisi missing values pada kolom categorical dengan modus.
kolom_kategori = ["Gender", "Company Type", "WFH Setup Available"]

for kolom in kolom_kategori:
    df_clean[kolom] = df_clean[kolom].fillna(df_clean[kolom].mode()[0])

# Cek ulang missing values.
df_clean.isna().sum()

## Visualisasi Setelah Data Dibersihkan

Bagian ini dibuat supaya notebook lebih mudah dipahami secara visual.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_clean["Burn Rate"], bins=25, kde=True, color="#4C72B0")
plt.title("Distribusi Burn Rate")
plt.xlabel("Burn Rate")
plt.ylabel("Jumlah Karyawan")
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))

for nomor, kolom in enumerate(fitur):
    plt.subplot(2, 2, nomor + 1)
    sns.histplot(df_clean[kolom], bins=20, kde=True, color="#55A868")
    plt.title("Distribusi " + kolom)
    plt.xlabel(kolom)
    plt.ylabel("Jumlah")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df_clean[fitur + ["Burn Rate"]].corr(), annot=True, cmap="viridis")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(15, 4))

for nomor, kolom in enumerate(kolom_kategori):
    plt.subplot(1, 3, nomor + 1)
    sns.boxplot(data=df_clean, x=kolom, y="Burn Rate")
    plt.title("Burn Rate berdasarkan " + kolom)
    plt.xlabel(kolom)
    plt.ylabel("Burn Rate")

plt.tight_layout()
plt.show()

## 4. Data Clustering

Pada tahap ini kita menggunakan **K-Means**.

Requirement project:

- Algoritma: K-Means
- Jumlah cluster: 2
- Hasil cluster ditambahkan sebagai kolom `Cluster`

In [ ]:
X = df_clean[fitur]

# Scaling agar semua fitur memiliki skala yang seimbang.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-Means dengan 2 cluster.
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df_clean["Cluster"] = kmeans.fit_predict(X_scaled)

df_clean[["Designation", "Resource Allocation", "Mental Fatigue Score", "Years Working", "Burn Rate", "Cluster"]].head()

In [ ]:
# Melihat jumlah data pada setiap cluster.
jumlah_cluster = df_clean["Cluster"].value_counts().sort_index()
jumlah_cluster

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(x=jumlah_cluster.index, y=jumlah_cluster.values)
plt.title("Jumlah Karyawan pada Setiap Cluster")
plt.xlabel("Cluster")
plt.ylabel("Jumlah Karyawan")
plt.show()

In [ ]:
# Melihat rata-rata Burn Rate dan Mental Fatigue Score pada setiap cluster.
ringkasan_cluster = df_clean.groupby("Cluster")[["Burn Rate", "Mental Fatigue Score", "Resource Allocation"]].mean()
ringkasan_cluster

In [ ]:
# Cluster dengan rata-rata Burn Rate lebih tinggi dianggap High Burnout Risk.
urutan_cluster = ringkasan_cluster.sort_values("Burn Rate").index.tolist()

label_risiko = {
    urutan_cluster[0]: "Low Burnout Risk",
    urutan_cluster[1]: "High Burnout Risk"
}

df_clean["Risk Label"] = df_clean["Cluster"].map(label_risiko)

ringkasan_cluster_tampil = ringkasan_cluster.copy()
ringkasan_cluster_tampil["Risk Label"] = ringkasan_cluster_tampil.index.map(label_risiko)
ringkasan_cluster_tampil

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_clean,
    x="Resource Allocation",
    y="Mental Fatigue Score",
    hue="Risk Label",
    size="Burn Rate",
    sizes=(30, 180),
    alpha=0.7
)
plt.title("Hasil Clustering Risiko Burnout")
plt.xlabel("Resource Allocation")
plt.ylabel("Mental Fatigue Score")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x="Risk Label", y="Burn Rate")
plt.title("Perbandingan Burn Rate pada Setiap Risiko")
plt.xlabel("Risk Label")
plt.ylabel("Burn Rate")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=ringkasan_cluster_tampil.reset_index(),
    x="Risk Label",
    y="Mental Fatigue Score"
)
plt.title("Rata-rata Mental Fatigue Score per Risiko")
plt.xlabel("Risk Label")
plt.ylabel("Rata-rata Mental Fatigue Score")
plt.show()

## 5. Regression

Target variabel untuk regression adalah `Cluster`, yaitu hasil dari proses clustering.

Model yang digunakan adalah `RandomForestRegressor`.

In [ ]:
X = df_clean[fitur]
y = df_clean["Cluster"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

regression_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1)
regression_model.fit(X_train, y_train)

hasil_prediksi_regression = regression_model.predict(X_test)

print("Training regression selesai.")

## 6. Classification

Target variabel untuk classification juga `Cluster`.

Model yang digunakan adalah `RandomForestClassifier`.

In [ ]:
classification_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
classification_model.fit(X_train, y_train)

hasil_prediksi_classification = classification_model.predict(X_test)

print("Training classification selesai.")

## 7. Evaluation

Pada tahap ini kita mengevaluasi model.

Regression menggunakan:

- MAE
- MSE
- R2 Score

Classification menggunakan:

- Accuracy
- Confusion Matrix
- Classification Report

In [ ]:
mae = mean_absolute_error(y_test, hasil_prediksi_regression)
mse = mean_squared_error(y_test, hasil_prediksi_regression)
r2 = r2_score(y_test, hasil_prediksi_regression)

hasil_regression = pd.DataFrame({
    "Metric": ["MAE", "MSE", "R2 Score"],
    "Value": [mae, mse, r2]
})

hasil_regression

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(data=hasil_regression, x="Metric", y="Value")
plt.title("Evaluation Regression")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, hasil_prediksi_regression, alpha=0.5, color="#4C72B0")
plt.title("Actual vs Predicted Cluster - Regression")
plt.xlabel("Actual Cluster")
plt.ylabel("Predicted Cluster")
plt.show()

In [ ]:
accuracy = accuracy_score(y_test, hasil_prediksi_classification)

print("Classification Evaluation")
print("Accuracy:", accuracy)
print()
print("Classification Report:")
print(classification_report(y_test, hasil_prediksi_classification))

In [ ]:
cm = confusion_matrix(y_test, hasil_prediksi_classification)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Classification")
plt.xlabel("Predicted Cluster")
plt.ylabel("Actual Cluster")
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "Fitur": fitur,
    "Importance": classification_model.feature_importances_
})

feature_importance = feature_importance.sort_values("Importance", ascending=False)
feature_importance

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=feature_importance, x="Importance", y="Fitur")
plt.title("Feature Importance - Classification Model")
plt.xlabel("Importance")
plt.ylabel("Fitur")
plt.show()

## 8. Deployment Sederhana

Pada project utama, deployment bisa dibuat menggunakan Streamlit.

Di notebook ini kita buat simulasi prediksi sederhana menggunakan fungsi Python.

Input yang digunakan:

- Designation
- Resource Allocation
- Mental Fatigue Score
- Years Working

In [ ]:
def prediksi_burnout(designation, resource_allocation, mental_fatigue_score, years_working):
    data_baru = pd.DataFrame([{
        "Designation": designation,
        "Resource Allocation": resource_allocation,
        "Mental Fatigue Score": mental_fatigue_score,
        "Years Working": years_working
    }])

    cluster_prediksi = int(classification_model.predict(data_baru)[0])
    risiko = label_risiko[cluster_prediksi]

    if risiko == "High Burnout Risk":
        rekomendasi = "Kurangi beban kerja, perbaiki waktu istirahat, dan lakukan monitoring fatigue."
    else:
        rekomendasi = "Pertahankan pola kerja saat ini dan tetap pantau kondisi mental fatigue."

    return cluster_prediksi, risiko, rekomendasi

In [ ]:
# Contoh input prediksi.
contoh_input = {
    "Designation": 2,
    "Resource Allocation": 5,
    "Mental Fatigue Score": 6.5,
    "Years Working": 3
}

cluster, risiko, rekomendasi = prediksi_burnout(
    designation=contoh_input["Designation"],
    resource_allocation=contoh_input["Resource Allocation"],
    mental_fatigue_score=contoh_input["Mental Fatigue Score"],
    years_working=contoh_input["Years Working"]
)

print("Predicted cluster:", cluster)
print("Burnout risk:", risiko)
print("Recommendation:", rekomendasi)

In [ ]:
contoh_df = pd.DataFrame({
    "Fitur": list(contoh_input.keys()),
    "Nilai Input": list(contoh_input.values()),
    "Rata-rata Dataset": [df_clean[kolom].mean() for kolom in contoh_input.keys()]
})

contoh_df

In [ ]:
plt.figure(figsize=(9, 5))

nomor_fitur = np.arange(len(contoh_df))
lebar_bar = 0.35

plt.bar(nomor_fitur - lebar_bar / 2, contoh_df["Nilai Input"], width=lebar_bar, label="Nilai Input")
plt.bar(nomor_fitur + lebar_bar / 2, contoh_df["Rata-rata Dataset"], width=lebar_bar, label="Rata-rata Dataset")

plt.title("Perbandingan Input Baru dengan Rata-rata Dataset")
plt.xlabel("Fitur")
plt.ylabel("Nilai")
plt.xticks(nomor_fitur, contoh_df["Fitur"], rotation=30, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## Kesimpulan

Berdasarkan framework CRISP-DM, notebook ini sudah melakukan:

- Business understanding
- Data understanding
- Data preparation
- Visualisasi missing values, kategori, distribusi, korelasi, dan cluster
- Clustering dengan K-Means 2 cluster
- Regression dengan RandomForestRegressor
- Classification dengan RandomForestClassifier
- Evaluation model dengan tabel dan grafik
- Deployment sederhana menggunakan fungsi prediksi

Kesimpulan utama:

- Cluster dibuat berdasarkan fitur numerik karyawan.
- Cluster dengan rata-rata `Burn Rate` lebih tinggi diberi label **High Burnout Risk**.
- Model classification digunakan untuk memprediksi cluster dari input baru.
- Visualisasi membantu melihat pola data dan hasil model dengan lebih jelas.